In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
# Define where we save our scraped data
RAW_DATA_PATH = "../data/raw/"
os.makedirs(RAW_DATA_PATH, exist_ok=True)

print(f"Data will be saved to: {RAW_DATA_PATH}")

Data will be saved to: ../data/raw/


In [3]:
def scrape_chittorgarh_ipos():
    """
    Scrapes IPO listing data from Chittorgarh.
    Returns a raw DataFrame with all available IPO records.
    """

    url = "https://www.chittorgarh.com/report/ipo-subscription-status-live-bidding-data/93/"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    print("Sending request to Chittorgarh...")
    response = requests.get(url, headers=headers, timeout=15)

    print(f"Response status code: {response.status_code}")

    if response.status_code != 200:
        print("Failed to fetch page. Try again.")
        return None

    soup = BeautifulSoup(response.content, "html.parser")

    # Find all tables on the page
    tables = soup.find_all("table")
    print(f"Found {len(tables)} tables on the page")

    return soup, tables

# Run the scraper
soup, tables = scrape_chittorgarh_ipos()

Sending request to Chittorgarh...
Response status code: 200
Found 0 tables on the page


In [4]:
def extract_ipo_table(tables):
    """
    Loops through all tables found on the page
    and extracts the one that contains IPO data.
    """

    all_dataframes = []

    for i, table in enumerate(tables):
        try:
            df = pd.read_html(str(table))[0]
            print(f"Table {i}: {df.shape[0]} rows x {df.shape[1]} cols | Columns: {list(df.columns)}")
            all_dataframes.append(df)
        except Exception as e:
            print(f"Table {i}: Could not parse — {e}")

    return all_dataframes

all_dfs = extract_ipo_table(tables)

In [5]:
# Check how many tables were found
print(f"Total tables found: {len(all_dfs)}")

# Preview each table's columns and first row
for i, df in enumerate(all_dfs):
    print(f"\n--- Table {i} ---")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(2))
    print("-" * 50)

Total tables found: 0


In [6]:
import pandas as pd

# Load both files
df_ipo = pd.read_csv("../data/raw/ipo_2010_2021.csv")
df_nifty = pd.read_csv("../data/raw/NIFTY_2010_2021.csv")

print("=== IPO Dataset ===")
print(f"Shape: {df_ipo.shape}")
print(f"Columns: {list(df_ipo.columns)}")
df_ipo.head(3)

=== IPO Dataset ===
Shape: (264, 14)
Columns: ['Date', 'IPO Name', 'Profile', 'Issue Size (in crores)', 'QIB', 'HNI', 'RII', 'Total', 'Issue', 'Listing Open', 'Listing Close', 'Listing Gains(%)', 'CMP', 'Current  Gains (%)']


,Date,IPO Name,Profile,Issue Size (in crores),QIB,HNI,RII,Total,Issue,Listing Open,Listing Close,Listing Gains(%),CMP,Current Gains (%)
0,29-07-21,Tatva Chintan,https://www.moneycontrol.com/ipo/tatva-chintan...,500.0,2.55,9.78,13.36,9.50,1083.0,2111.8,2310.25,113.32,"2,268.50",109.46
1,23-07-21,Zomato,https://www.moneycontrol.com/ipo/zomato_Z01.html,9375.0,51.79,32.96,7.45,38.25,76.0,115.0,125.85,65.59,133.35,75.46
2,19-07-21,Clean Science,https://www.moneycontrol.com/ipo/clean-science...,1546.0,156.37,206.43,9.00,93.41,900.0,1784.4,1585.20,76.13,"1,682.80",86.98


In [7]:
print("=== NIFTY Dataset ===")
print(f"Shape: {df_nifty.shape}")
print(f"Columns: {list(df_nifty.columns)}")
df_nifty.head(3)

=== NIFTY Dataset ===
Shape: (2873, 7)
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Shares Traded', 'Turnover (Rs. Cr)']


,Date,Open,High,Low,Close,Shares Traded,Turnover (Rs. Cr)
0,04-Jan-2010,5200.90,5238.45,5167.10,5232.2,148652424,6531.61
1,05-Jan-2010,5277.15,5288.35,5242.40,5277.9,240844424,7969.62
2,06-Jan-2010,5278.15,5310.85,5260.05,5281.8,216147837,7892.60


In [8]:
# Save raw data confirmation
print("Raw data loaded successfully!")
print(f"IPO records: {df_ipo.shape[0]}")
print(f"NIFTY records: {df_nifty.shape[0]}")

# Save copies to raw folder for reference
df_ipo.to_csv("../data/raw/ipo_raw_backup.csv", index=False)
df_nifty.to_csv("../data/raw/nifty_raw_backup.csv", index=False)
print("Backup saved to data/raw/")

Raw data loaded successfully!
IPO records: 264
NIFTY records: 2873
Backup saved to data/raw/
